## Weather Data Collection — Vancouver

Pulls Vancouver's coordinates via OpenWeatherMap's Geocoding API, then uses those 
coordinates to fetch 4 years of daily historical weather (temperature, rainfall) 
from Open-Meteo. This data pairs with Google Trends search interest 
(see `pytrends.ipynb`) to test whether weather drives outdoor-apparel demand.

In [2]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv

### Get Vancouver's coordinates

OpenWeatherMap's Geocoding API converts a city name into latitude/longitude, which 
downstream weather APIs require.

In [3]:
load_dotenv()

openweather_api_key = os.getenv("OPENWEATHER_API_KEY")

def get_coordinates(city, state_code, country_code, limit=1):
    url = 'https://api.openweathermap.org/geo/1.0/direct'
    params = {
        'q': f'{city},{state_code},{country_code}',
        'limit': limit,
        'appid': openweather_api_key
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

vancouver = get_coordinates('Vancouver','BC','CA')
print(vancouver[0].keys())

dict_keys(['name', 'local_names', 'lat', 'lon', 'country', 'state'])


Extract lat/lon from the geocoding response for use in the weather API call below.

In [4]:
lat = vancouver[0]['lat']
lon = vancouver[0]['lon']
print('Vancouver Coordinates:',lat,',',lon)

Vancouver Coordinates: 49.2608724 , -123.113952


### Pull historical daily weather

Historical weather is sourced from Open-Meteo.

Covers January 1, 2022 – December 31, 2025, aligned with the date range used for the Google Trends pull to ensure both datasets can be joined cleanly downstream.

In [5]:
def get_historical_weather(lat, lon, start_date, end_date, timezone="America/Vancouver"):
    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean", "rain_sum"],
        "timezone": timezone
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

weather_data = get_historical_weather(lat, lon, "2022-01-01", "2025-12-31")

Optional display settings for viewing the full daily dataset in this notebook commented out by default since the exported CSV/SQL table is the actual data 
source used downstream.

In [6]:
# to view the entire scrollable table
# pd.set_option('display.max_rows', None)

# to view original display format
# pd.reset_option('display.max_rows')

### Build the weather DataFrame

Converts the API's nested `daily` response into a flat DataFrame one row per day, 
ready for export to CSV and import into MySQL for cleaning and weekly aggregation.

In [7]:
df = pd.DataFrame(data=weather_data['daily'])
df

,time,temperature_2m_max,temperature_2m_min,temperature_2m_mean,rain_sum
0,2022-01-01,0.9,-8.8,-3.9,0.0
1,2022-01-02,3.2,0.1,1.9,6.9
2,2022-01-03,2.6,-0.6,1.4,5.6
3,2022-01-04,2.3,-0.6,0.6,3.5
4,2022-01-05,0.1,-2.8,-1.5,0.1
...,...,...,...,...,...
1456,2025-12-27,2.6,-1.0,0.7,0.0
1457,2025-12-28,2.7,-0.2,1.0,0.0
1458,2025-12-29,6.1,0.3,2.8,0.0
1459,2025-12-30,4.2,-0.1,1.9,0.0
